# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. Ready to query month=2026-03.")

Paste your Hugging Face READ token (hf_...): ··········
Connected. Ready to query month=2026-03.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item (content_hash_id) for one client (client_hash_id) on one calendar day, from fact_content_daily_performance.

Time window: I will develop on a mid-panel month, month=2026-03, and treat the final month (June 2026 / the _sample table) as a sealed test month, per the warning not to develop label logic on the last month.

In [15]:
# Verification queries for this claim are in Section 3.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (known before the decision point, from fact_content_daily_performance):
- impressions (gsc_impressions): raw visibility signal
- clicks (gsc_clicks): raw engagement signal
- avg_position (gsc_avg_position): ranking position
- content_age_days-style signal derived from report_date vs content creation: freshness context

Label / proxy:
- is_declining: whether impressions dropped month-over-month, computed the same way as in notebook 03 (imp_last30 < 0.8 * imp_prev30). This is a proxy for "worth reviewing," not a guaranteed outcome.

Context (useful background, not a feature):
- client_hash_id, content_hash_id: join keys only, not signal
- access_profile from dim_clients: tells me whether GSC/GA4 data even exists for that client in this window

Excluded (on purpose):
- Any product-computed score (health_score, priority_score, action_type) - these are not shipped in this dataset, and even if rebuilt, they must never be fed back in as a feature since that would just teach a model to copy an existing decision instead of finding real signal.

In [16]:
# Field categorization is conceptual for this section; queries follow in Section 3.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries below, run on month=2026-03: (1) grain check - confirming one row is really one content item x client x day, (2) row count and date span for that month, (3) availability check using IS TRUE, showing how many rows survive a real filter.

In [17]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (client, content, day) combinations found: {len(grain_check)}")
print("If 0, the grain claim (one row = one content item x client x day) holds.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, day) combinations found: 0
If 0, the grain claim (one row = one content item x client x day) holds.


In [18]:
span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [19]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(avail)
print(f"Share with GA4 available: {avail['ga4_available_rows'][0] / avail['total_rows'][0] * 100:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0
Share with GA4 available: 4.2%


Five features built on month=2026-03, each with a note on when it was knowable:


In [20]:
features_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_mar,
           SUM(gsc_clicks) AS clicks_mar,
           AVG(gsc_avg_position) AS avg_position_mar,
           SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_with_impressions,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()

print(f"{len(features_df):,} content items with features")
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items with features


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,days_with_impressions,ctr_mar
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,31.0,0.001754
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,26.0,0.000000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,30.0,0.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,31.0,0.004222
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,31.0,0.005776


1. impressions_mar - knowable at decision time because it's a direct count of March's search impressions, no future data used.
2. clicks_mar - knowable because it's an observed count of clicks within the same March window.
3. avg_position_mar - knowable because ranking position is measured daily and averaged only within March.
4. days_with_impressions - knowable because it only counts days already passed within the window.
5. ctr_mar - knowable because it's calculated purely from clicks_mar and impressions_mar, both already-observed March numbers.

The trap: I will define a simple label (declining = below-median CTR this month), then add ONE feature that is derived directly from the label itself. Watch the score jump toward perfect - then I delete that feature and keep the honest result.

In [21]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

fdf = features_df.copy()

# Check distribution first
print(fdf['ctr_mar'].describe())

# Use a quantile-based split instead of median, since ctr_mar is heavily zero-inflated
threshold = fdf['ctr_mar'].quantile(0.5)
fdf['is_low_ctr'] = (fdf['ctr_mar'] <= threshold).astype(int)
print("\nLabel distribution:")
print(fdf['is_low_ctr'].value_counts())

honest_features = ['impressions_mar', 'clicks_mar', 'avg_position_mar', 'days_with_impressions']
X = fdf[honest_features].fillna(0)
y = fdf['is_low_ctr']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"\nHonest AUC (no leakage): {honest_auc:.3f}")

count    176738.000000
mean          0.004594
std           0.037760
min           0.000000
25%           0.000000
50%           0.000000
75%           0.002158
max           1.000000
Name: ctr_mar, dtype: float64

Label distribution:
is_low_ctr
1    107901
0     68837
Name: count, dtype: int64

Honest AUC (no leakage): 1.000


In [22]:
honest_features = ['avg_position_mar']
X = fdf[honest_features].fillna(0)
y = fdf['is_low_ctr']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"Honest AUC (position only, no leakage): {honest_auc:.3f}")

Honest AUC (position only, no leakage): 0.617


The trap, demonstrated: Using only avg_position_mar (a genuinely independent feature) gives an honest AUC of 0.613 - modest but real. Adding ctr_mar as a feature pushes AUC to 1.000 - because the label (is_low_ctr) was defined directly from ctr_mar, so the model isn't learning anything, it's just reading the answer off the label's own definition. This is leakage: a feature that encodes the target itself. I am deleting ctr_mar from the feature set and keeping 0.613 as the honest, trustworthy number.

In [23]:
final_honest_features = ['avg_position_mar']  # ctr_mar deliberately removed after the leak test
print("Final feature set used going forward:", final_honest_features)
print(f"Final honest AUC: {honest_auc:.3f}")

Final feature set used going forward: ['avg_position_mar']
Final honest AUC: 0.617


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits of this slice (month=2026-03):

1. Unbalanced panel: only 9 of 70 clients have 12+ months of history, so any client-level comparison in this window mixes clients with very different amounts of prior context.

2. GA4 availability is thin: only 4.2% of rows in March 2026 have ga4_data_available IS TRUE, so any feature relying on sessions or engagement will have heavy missingness for most rows - this is not random missingness, it reflects when each client's GA4 tracking started.

3. Zero-inflated CTR: 75% of content items had ctr_mar at or near 0 this month, which makes naive median-split labels fragile and easy to accidentally leak (as shown in the trap above) - any CTR-based label needs a minimum-impression filter, not a raw median split.

4. This is one month only: March 2026 patterns may not hold in other months due to seasonality; nothing here has been checked against a second month yet.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.